# Lab 3 — one controlled change and a defensible recommendation

100 points · 6% NP · same pairs as Lab 2 · due Thu 29 October 2026, 12:30 Santiago.

Lab 1 checked data and simple rules. Lab 2 changed one input under fixed comparison conditions.
Here you keep the inputs fixed and change one model setting, then explain errors and write for a
Head of Strategy. The supplied code runs the experiment; you must supply the checked explanation,
one-change specification, three supported failure patterns, memo and truthful AI record.

Follow the numbered headings. See README for setup and GLOSSARY for terms.
A losing additional candidate can earn full credit. The unchanged starter is incomplete.


## Environment and data check

Extract the whole ZIP and choose your kernel before running this block. Follow README.md if a check fails.

In [1]:
from pathlib import Path
import sys
import importlib
print('Selected Python:', sys.executable)
print('Version:', sys.version.split()[0], '| working directory:', Path.cwd())
required_files = ['lab03_support_v3.py', 'pitwindow_student_v2.py', 'data/learning_v2.csv']
missing = [name for name in required_files if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Extract the complete ZIP and open its folder as cwd. Missing: ' + ', '.join(missing))
for module in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']:
    loaded = importlib.import_module(module)
    print(module, loaded.__version__)
print('Ready: local files/imports found. Next cell checks data integrity and split.')


Selected Python: /home/m1lvn/UDD/IA_Workshop/labs/lab_03/.venv/bin/python
Version: 3.12.14 | working directory: /home/m1lvn/UDD/IA_Workshop/labs/lab_03
numpy 2.0.2
pandas 2.3.1


sklearn 1.8.0
scipy 1.17.1
joblib 1.4.2
Ready: local files/imports found. Next cell checks data integrity and split.


In [2]:
from pathlib import Path
import joblib
import pandas as pd
# Determinism: forest predict_proba (n_jobs=2) sums trees in thread order, so a window scored exactly at the
# selected threshold could flip between the comparison and the prediction export (1e-16 difference).
# Running joblib sequentially changes no model setting and makes every export agree.
joblib.parallel_config(backend='sequential')
from lab03_support_v3 import (
    FEATURES, SEED, BUDGET, load_learning, run_base_comparison,
    run_one_change, export_run, markdown_table,
)
ROOT = Path('.')
train, dev = load_learning(ROOT)
print('Seed:', SEED, '| development alert budget:', BUDGET)
print('Permitted inputs:', FEATURES)
audit = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive_windows': int(df.y.sum()),
     'eligible_events': df.loc[df.y.eq(1), 'event_id'].nunique(),
     'first_year': int(df.year.min()), 'last_year': int(df.year.max())}
    for name, df in [('train', train), ('dev', dev)]
])
display(audit)
assert set(train.opportunity_id).isdisjoint(dev.opportunity_id)


Seed: 414 | development alert budget: 0.1
Permitted inputs: ['circuit_id', 'compound', 'tyre_life', 'lap_number', 'progress', 'prior_pit_entries', 'pace_median_s', 'pace_slope_s_per_lap', 'pace_n']


,split,rows,positive_windows,eligible_events,first_year,last_year
0,train,63165,5523,1934,2018,2021
1,dev,9451,1073,370,2022,2022


## 1. Define the prediction

**Your answer:**

- **Prediction time.** One opportunity (window) is created at the end of a rival driver's lap L (`cut_s`, race time at that lap end). The model may only use what is recorded up to that moment.
- **Positive label.** `y = 1` if that same rival is observed entering the pit lane during their own next three laps (L+1 … L+3); otherwise `y = 0`. It is an *observed* entry by the rival, not an optimal strategy call for our own car. Several consecutive windows can point to the same entry (`event_id`), which is why we count both windows and distinct events. In the data, every positive window has `event_s > cut_s`, so the entry is always after the prediction time.
- **One allowed input.** `tyre_life`: the age in laps of the tyre set the rival is on at the end of lap L. It is known at the prediction time (it is last reported and can lag by a lap, as GLOSSARY notes). It is one of the nine `FEATURES` passed to the pipeline.
- **One future-known field kept out of X.** `event_s` (and the related `event_id`) records *when* the pit entry happened. It exists only after the entry, so it is used only to audit the label and count events; it never enters `FEATURES`. If it were an input, the model would effectively read the answer (leakage). `horizon_end_s` and `future_neutralisation` are also future-known and excluded.
- **What this retrospective snapshot can and cannot establish.** It is a corrected historical reconstruction (FastF1 timing, `manifest_lab03_v3.json`). It can show that, as recorded afterwards, each input describes the state at or before lap L, and that train (2018–2021) precedes development (2022, rounds 1–11). It cannot establish that these values would arrive on time in a live feed (latency, later corrections or missing laps, e.g. `pace_median_s` is missing in about 2% of rows). It also cannot show how the alert would behave in races after round 11 of 2022 or in a prospective trial.

## Practice reading the counts

Illustrative manual example only, NOT F1 data. Two positive windows refer to event A; a third refers
to event B. The final window is negative and triggers a false alert.

| Window | y | Event | Alert |
|---|---:|---|---|
| w1 | 1 | A | yes |
| w2 | 1 | A | yes |
| w3 | 1 | B | no |
| w4 | 0 | none | yes |

Ungraded check: count TP, FP, FN, alerts, eligible/caught events, precision and event recall.
What changes if w2 stops alerting? Think first, then open the self-check below.


### Optional practice response

Before opening the self-check below, write or think through: TP = __; FP = __; FN = __; alerts = __; eligible/caught events = __ / __; precision = __; event recall = __. If w2 becomes quiet: __. This exercise is ungraded; no separate submission is required.

<details>
<summary>Open after trying: self-check for the four fictional windows</summary>

TP=2, FP=1, FN=1, TN=0; alerts=3; precision=2/3. There are two eligible events A/B, one caught, so event recall=1/2. If w2 becomes quiet: TP=1, FP=1, FN=2, alerts=2, precision=1/2; A is still caught by w1 and event recall remains 1/2.

</details>

In [3]:
base_comparison, fitted, base_policies = run_base_comparison(train, dev)
compact_columns = ['model', 'alerts', 'precision', 'events_caught', 'eligible_events', 'event_recall']
print('Start with DEVELOPMENT. Undefined precision means there were no alerts.')
display(base_comparison.loc[base_comparison.split.eq('dev'), compact_columns])
columns = ['model', 'split', 'n', 'TP', 'FP', 'FN', 'TN', 'alerts', 'alert_rate',
           'precision', 'events_caught', 'eligible_events', 'event_recall', 'threshold']
print("Audit view: train and development reuse the threshold selected for that model on dev.")
display(base_comparison[columns])


Start with DEVELOPMENT. Undefined precision means there were no alerts.


,model,alerts,precision,events_caught,eligible_events,event_recall
1,never_alert,0,NaN,0,370,0.000000
3,historical_baseline,931,0.268528,94,370,0.254054
5,tree,571,0.269702,63,370,0.170270
7,forest_all,938,0.276119,107,370,0.289189
9,forest_sqrt,945,0.359788,140,370,0.378378


Audit view: train and development reuse the threshold selected for that model on dev.


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
0,never_alert,train,63165,0,0,5523,57642,0,0.000000,NaN,0,1934,0.000000,NaN
1,never_alert,dev,9451,0,0,1073,8378,0,0.000000,NaN,0,370,0.000000,NaN
2,historical_baseline,train,63165,3042,6645,2481,50997,9687,0.153360,0.314029,1137,1934,0.587901,0.161926
3,historical_baseline,dev,9451,250,681,823,7697,931,0.098508,0.268528,94,370,0.254054,0.161926
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
6,forest_all,train,63165,2503,4503,3020,53139,7006,0.110916,0.357265,969,1934,0.501034,0.718176
7,forest_all,dev,9451,259,679,814,7699,938,0.099249,0.276119,107,370,0.289189,0.718176
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283


## 2. Explain the common comparison

**Development first (9,451 windows, 1,073 positive windows, 370 eligible pit entries; budget = at most 945 alerts).**

| System | Alerts (rate) | TP / alerts = precision | Events caught / eligible = event recall | Threshold |
|---|---:|---:|---:|---:|
| never_alert | 0 (0%) | 0 / 0 = undefined | 0 / 370 = 0.000 | none |
| historical_baseline | 931 (9.9%) | 250 / 931 = 0.269 | 94 / 370 = 0.254 | 0.162 |
| tree | 571 (6.0%) | 154 / 571 = 0.270 | 63 / 370 = 0.170 | 0.714 |
| forest_all | 938 (9.9%) | 259 / 938 = 0.276 | 107 / 370 = 0.289 | 0.718 |
| forest_sqrt | 945 (10.0%) | 340 / 945 = 0.360 | 140 / 370 = 0.378 | 0.608 |

**forest_all vs forest_sqrt.** With almost the same number of alerts (938 vs 945, +7), forest_sqrt catches 33 more distinct entries (140 vs 107) and issues fewer false alerts (605 vs 679 FP). It is the only learned model clearly above the historical baseline (94/370). The two forests are identical except `max_features`. After preprocessing, each tree sees 48 columns (checkable with `len(fitted['forest_sqrt'][:-1].get_feature_names_out())`): 7 numeric inputs, plus missing-value indicators and the one-hot columns for `circuit_id` and `compound`. forest_all (`max_features=1.0`) may choose among all 48 at every split. forest_sqrt (`'sqrt'`) draws a fresh random set of int(√48) = 6 candidate columns at each split. No column is removed: every column remains available at other splits and in other trees. Our hypothesis, not measured here: when all 48 are always available, the strongest numeric inputs tend to win the same top splits in most trees, so the 150 trees look alike. Restricting the candidates forces other splits and makes the averaged score rank windows differently. We did not measure tree similarity, so this explains a plausible mechanism, not a proven cause.

**Reading the counts.**
- *Undefined precision:* never_alert issues 0 alerts, so precision is 0/0 = undefined, not 0. Its event recall is 0/370.
- *TP windows are not caught events:* forest_sqrt has 340 TP windows but 140 caught entries, about 2.4 alerted positive windows per caught entry. The 370 entries have 1,073 positive windows, because one pit entry is inside the 3-lap horizon of up to three consecutive laps. Event recall counts each entry once if *any* of its windows alerts.
- *Same policy, different thresholds:* each model uses the same rule on dev: at most 10% alerts, maximise distinct entries caught, with precision then threshold as tie-breakers. Scores are on different scales, so the numeric cutoffs differ (0.608 vs 0.718). Ties also matter: the depth-3 tree produces only 8 distinct scores. Its next lower score level would alert 1,898 windows, over the 945 limit, so it stops at 571 and leaves budget unused. Comparing systems compares *model + selected threshold*, not the model alone.

**One train/dev gap (forest_sqrt).** Event recall is 1,005/1,934 = 0.520 on train vs 140/370 = 0.378 on dev. This gap is not a clean measure of overfitting:
1. Train scores are in-sample (the forest was fit on those windows).
2. The dev-selected threshold, reused unchanged on train, alerts 14.4% of train windows (9,085), because the 10% budget applies only to dev. More alerts mechanically catch more entries.
3. Precision moves the other way (train 0.294 vs dev 0.360), which a simple "memorised train" story would not predict.

Dev itself was used to choose thresholds and to compare systems, so its numbers are also somewhat optimistic and are not an untouched final test. The gap alone cannot tell us its cause (season change, in-sample fit or the different alert rate).

## 3. Record your expectation before your additional run

The base outputs are already supplied. Complete `experiment_spec.md` before running YOUR candidate
for the first time. If already run, say so and keep a retrospective account. Do not invent history.

Choose ONE tuple: tree / max_depth / 6 (old 3); forest_sqrt / max_depth / 3 (old 6);
or forest_sqrt / min_samples_leaf / 100 (old 30). No search for the best of all three.
The helper clones the base pipeline and changes one setting. Its threshold is reselected on dev
under the same policy, so the numeric threshold may differ. This compares the resulting alert systems.


Choose one additional configuration. Repeating that same configuration for debugging or Restart + Run All is allowed. Disclose any different configurations you already tried; do not hide a search.

In [4]:
EXPERIMENT_MODEL_ID = 'forest_sqrt'  # 'tree' or 'forest_sqrt', matching the allowed table
EXPERIMENT_PARAMETER = 'min_samples_leaf'  # 'max_depth' or 'min_samples_leaf'
EXPERIMENT_VALUE = 100      # the one allowed new value (old 30)

In [5]:
change = None
policies = dict(base_policies)
comparison = base_comparison.copy()
if any(value is None for value in [EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE]):
    print('INCOMPLETE STARTER: configure one additional experiment in extra-config.')
else:
    extra_rows, extra_model, extra_policy, change = run_one_change(
        train, dev, fitted, EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE)
    comparison = pd.concat([base_comparison, extra_rows], ignore_index=True)
    policies[change['model_id']] = extra_policy
    display(comparison.loc[comparison.model.isin([change['parent'], change['model_id']]), columns])


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283
10,forest_sqrt__min_samples_leaf_100,train,63165,2716,6900,2807,50742,9616,0.152236,0.282446,1010,1934,0.522234,0.601914
11,forest_sqrt__min_samples_leaf_100,dev,9451,376,562,697,7816,938,0.099249,0.400853,159,370,0.429730,0.601914


## 4. Explain your additional result

**Candidate:** `forest_sqrt__min_samples_leaf_100` (only `min_samples_leaf` changed, 30 → 100; `changed_parameters` in `run_manifest_v3.json`). Development numbers vs the unchanged parent:

| Development (9,451 windows, 370 eligible entries) | Parent `forest_sqrt` | Candidate (leaf 100) | Change |
|---|---:|---:|---:|
| Alerts (alert rate) | 945 (10.0%) | 938 (9.9%) | −7 |
| Entries caught / eligible = event recall | 140 / 370 = 0.378 | 159 / 370 = 0.430 | +19 entries |
| TP / alerts = precision | 340 / 945 = 0.360 | 376 / 938 = 0.401 | +36 TP |
| False alerts (FP) | 605 | 562 | −43 |
| Selected threshold | 0.608 | 0.602 | −0.006 |
| AP (threshold-free, supplementary) | 0.331 | 0.344 | +0.013 |
| Train event recall at the same threshold (alert rate) | 1,005 / 1,934 = 0.520 (14.4%) | 1,010 / 1,934 = 0.522 (15.2%) | |
| Train − dev event-recall gap | 0.142 | 0.092 | −0.050 |

**Did the expectation hold? Partly.**
- *Held:* the train/dev gap shrank (0.142 → 0.092). Train AP fell from 0.310 to 0.297 while dev AP rose. That matches "less train-specific scores": the candidate fits its own training windows slightly less and ranks dev windows slightly better.
- *Wrong:* we expected about 140 ± 10 entries and "no clear improvement". The candidate caught 159 (+19), outside our range.
- *Wrong:* we expected more ties to move the threshold. The threshold barely moved (0.608 → 0.602). The candidate actually has slightly *more* distinct dev scores (8,096 vs 7,815), so ties were not the story.

**Model vs alert threshold.** The candidate used 7 *fewer* alerts, so the extra 19 entries do not come from spending more of the budget. It comes from a different ordering of windows: 36 more of the alerted windows are true positives. The higher dev AP, which does not depend on any threshold, points the same way. Still, each system is "fitted model + its own dev-selected threshold", so we compare the two resulting alert systems, not the parameter in isolation.

**Plausible mechanism (observed vs hypothesis).**
- *Observed:* requiring at least 100 training windows per leaf cut the mean number of leaves per tree from 40.6 to 32.3. Checkable with `np.mean([t.get_n_leaves() for t in extra_model[-1].estimators_])`, and the same expression on `fitted['forest_sqrt']`.
- *Hypothesis (not measured):* the removed leaves were small 2018–2021 situations, for example a specific circuit or tyre-age corner. Their scores did not transfer to 2022. With larger leaves, each score averages more past windows, so the ranking relies more on broad signals such as tyre age, race progress and previous stops. We did not inspect which leaves disappeared, and the 2022 rule changes are a possible confounder.

**Bounded recommendation.** Within this protocol, the leaf-100 forest is the strongest of the six alert systems on development: 159/370 entries with 938 alerts, against 140/370 for its parent and 94/370 for the historical baseline. We would carry it forward as the *candidate to verify*, not as a validated tool. Limits:
1. Development was used to select each threshold *and* to pick this winner among six systems, so 0.430 is optimistic. A +19-entry difference on one half-season could partly be noise; we ran a single seed and no uncertainty estimate.
2. This is a retrospective snapshot. It does not show that inputs arrive on time live, or that alerts translate into positions or seconds.
3. The scores are not calibrated probabilities. 6 of 10 alerts are still false (562/938).

The next check is a single, pre-registered evaluation on later untouched data (e.g., the held-out 2023–2024 test), with the threshold frozen exactly as selected here.

## 5. Explain three distinct failure patterns

Keep your additional candidate and its selected threshold frozen. Inspect the supplied tables, choose the denominator for your claim, and use the queries to find supporting cases. This step does not train or retune.

### Choose your denominator

| Question within your context | Calculation | Denominator counts |
|---|---|---|
| Of the alerts issued, how many were false? | FP / (TP + FP) | Alerted windows; false-alert fraction, equal to 1 − precision when defined. |
| Of the negative windows, how many alerted? | FP / (FP + TN) | Negative windows; false positive rate (FPR). |
| Of the positive windows, how many were quiet? | FN / (TP + FN) | Positive windows; `miss_rate` in the context export. |
| Of the eligible events, how many were completely missed? | Completely missed events / eligible events | Distinct pit entries; use all positive windows for each event. |

A zero denominator means undefined, shown as NaN; it does not mean zero errors. Name your population and unit with every fraction. Window context groups do not define an event population automatically: if describing missed events in a context, state which distinct events you include and check all their positive windows, including those outside that context. The event query below checks a whole event.


### A complete entry from the four fictional windows

This uses only the four-window practice example, not the Rival dataset or an additional candidate's results.

“For the context of all four toy windows, w4 is a false alert (y=0, alert=yes). One of the three alerts is false: FP/alerts=1/3. The traceable case is `opportunity_id=w4`. This could prompt an unnecessary review of a rival that does not enter within the horizon; next check the time and outcome around w4 before attributing a cause. With only four invented windows, this illustrates the reasoning format and cannot establish a general effect.”

The same counts give FPR=1/(1+0)=1/1 over negative windows, and miss_rate=1/(2+1)=1/3 over positive windows. Those fractions answer different questions. Event B has one positive window (w3), none alerted: one of two eligible events is completely missed. A quiet w2 alone would not make event A missed because w1 still alerts.


In [6]:
from lab03_support_v3 import development_predictions, context_error_summary, event_error_summary
predictions = None
if change is None:
    print('INCOMPLETE: run your additional candidate before examining its errors.')
else:
    predictions = development_predictions(dev, extra_model, extra_policy, change['model_id'])
    context_errors = context_error_summary(predictions)
    event_errors = event_error_summary(predictions)
    print('Descriptive context tables. No refit or threshold change occurs here.')
    display(context_errors)
    print('FP window examples (highest scores first):')
    display(predictions.loc[predictions.error_type.eq('FP')].sort_values('score', ascending=False).head(8))
    print('FN window examples (lowest scores first):')
    display(predictions.loc[predictions.error_type.eq('FN')].sort_values('score').head(8))
    print('Missed events:', int(event_errors.missed_event.sum()), '/', len(event_errors))
    display(event_errors.loc[event_errors.missed_event].head(8))


Descriptive context tables. No refit or threshold change occurs here.


,dimension,context,n,positive_windows,alerts,TP,FP,FN,TN,precision,miss_rate
0,compound,HARD,4037,266,274,44,230,222,3541,0.160584,0.834586
1,compound,INTERMEDIATE,377,86,62,36,26,50,265,0.580645,0.581395
2,compound,MEDIUM,3593,473,421,189,232,284,2888,0.448931,0.600423
3,compound,SOFT,1211,201,138,82,56,119,954,0.594203,0.592040
4,compound,UNKNOWN,4,0,0,0,0,0,4,NaN,NaN
5,compound,WET,229,47,43,25,18,22,164,0.581395,0.468085
6,circuit_id,10,797,61,104,42,62,19,674,0.403846,0.311475
7,circuit_id,144,720,88,73,30,43,58,589,0.410959,0.659091
8,circuit_id,149,617,55,87,33,54,22,508,0.379310,0.400000
9,circuit_id,15,1032,159,35,19,16,140,857,0.542857,0.880503


FP window examples (highest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
3623,2022-05-08:3:23:5956.816000,2022-05-08,3,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.729027,True,FP
3626,2022-05-08:3:26:6242.013000,2022-05-08,3,26,151,MEDIUM,25.0,0.456140,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.715471,True,FP
557,2022-03-20:4:17:5503.620000,2022-03-20,4,17,63,MEDIUM,16.0,0.298246,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.710303,True,FP
560,2022-03-20:4:20:5810.495000,2022-03-20,4,20,63,MEDIUM,19.0,0.350877,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.708626,True,FP
3305,2022-05-08:11:20:5636.771000,2022-05-08,11,20,151,MEDIUM,19.0,0.350877,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.705372,True,FP
3625,2022-05-08:3:25:6147.017000,2022-05-08,3,25,151,MEDIUM,24.0,0.438596,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.702046,True,FP
4012,2022-05-08:77:21:5739.305000,2022-05-08,77,21,151,MEDIUM,20.0,0.368421,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701659,True,FP
3880,2022-05-08:55:23:5911.352000,2022-05-08,55,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701486,True,FP


FN window examples (lowest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2455,2022-04-24:16:51:8308.778000,2022-04-24,16,51,6,SOFT,1.0,0.809524,2022-04-24:16:pit:8477.893000,1.0,forest_sqrt__min_samples_leaf_100,0.168912,False,FN
8576,2022-07-10:16:56:7713.940000,2022-07-10,16,56,19,HARD,6.0,0.788732,2022-07-10:16:pit:7863.977000,1.0,forest_sqrt__min_samples_leaf_100,0.172992,False,FN
8517,2022-07-10:14:55:7702.648000,2022-07-10,14,55,19,HARD,0.0,0.774648,2022-07-10:14:pit:7850.818000,1.0,forest_sqrt__min_samples_leaf_100,0.175046,False,FN
8518,2022-07-10:14:56:7773.231000,2022-07-10,14,56,19,HARD,0.0,0.788732,2022-07-10:14:pit:7850.818000,1.0,forest_sqrt__min_samples_leaf_100,0.175046,False,FN
8577,2022-07-10:16:57:7782.783000,2022-07-10,16,57,19,HARD,7.0,0.802817,2022-07-10:16:pit:7863.977000,1.0,forest_sqrt__min_samples_leaf_100,0.175175,False,FN
8374,2022-07-10:1:55:7648.611000,2022-07-10,1,55,19,HARD,1.0,0.774648,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.176645,False,FN
8375,2022-07-10:1:56:7718.344000,2022-07-10,1,56,19,HARD,1.0,0.788732,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.179797,False,FN
8376,2022-07-10:1:57:7788.187000,2022-07-10,1,57,19,HARD,1.0,0.802817,2022-07-10:1:pit:7869.472000,1.0,forest_sqrt__min_samples_leaf_100,0.179797,False,FN


Missed events: 211 / 370


,event_id,race_id,driver_id,positive_windows,alerted_positive_windows,caught,missed_event
4,2022-03-20:11:pit:8037.416000,2022-03-20,11,3,0,False,True
6,2022-03-20:14:pit:6293.550000,2022-03-20,14,3,0,False,True
7,2022-03-20:14:pit:8006.856000,2022-03-20,14,3,0,False,True
10,2022-03-20:16:pit:8333.486000,2022-03-20,16,3,0,False,True
13,2022-03-20:18:pit:8372.487000,2022-03-20,18,2,0,False,True
16,2022-03-20:1:pit:8020.852000,2022-03-20,1,3,0,False,True
17,2022-03-20:1:pit:9348.693000,2022-03-20,1,3,0,False,True
20,2022-03-20:20:pit:8430.179000,2022-03-20,20,2,0,False,True


### Find the cases behind your claim

Set CONTEXT_COLUMN to a supplied context column (for example compound or circuit_id), CONTEXT_VALUE to an actual value of the same type shown in your predictions, and ERROR_KIND to FP or FN. The summary uses all rows in that context, so its denominators do not accidentally count only errors. `cases` contains every matching case; the display previews eight. For three patterns you can repeat the query with recorded settings, or copy this query block within this section. Keep each setting and relevant output alongside its written claim.

For a whole-event claim, copy an event_id from the missed-event table into EVENT_TO_INSPECT. The second query retrieves all its positive windows, including their opportunity_id values, without a context filter. An event is missed only if none alerts. Unknown IDs return no rows and are not evidence of a missed event.


In [7]:
from lab03_support_v3 import context_query, positive_windows_for_event

CONTEXT_COLUMN = 'compound'  # pattern 1: false alerts on HARD tyres
CONTEXT_VALUE = 'HARD'
ERROR_KIND = 'FP'

if predictions is None:
    print('Run your additional candidate first; its frozen predictions are needed.')
elif CONTEXT_COLUMN is None or CONTEXT_VALUE is None:
    print('Set the context column/value above to inspect your own error evidence.')
else:
    context_summary, cases = context_query(predictions, CONTEXT_COLUMN, CONTEXT_VALUE, ERROR_KIND)
    display(context_summary)
    print('Matching cases:', len(cases), '(first eight shown)')
    display(cases.head(8))


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,compound,HARD,4037,44,230,222,3541,274,3771,266,0.839416,0.060992,0.834586


Matching cases: 230 (first eight shown)


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
1068,2022-03-27:20:21:6034.524000,2022-03-27,20,21,149,HARD,22.0,0.42,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.653193,True,FP
1069,2022-03-27:20:22:6130.163000,2022-03-27,20,22,149,HARD,23.0,0.44,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.651535,True,FP
1070,2022-03-27:20:23:6225.905000,2022-03-27,20,23,149,HARD,24.0,0.46,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.624628,True,FP
1071,2022-03-27:20:24:6321.298000,2022-03-27,20,24,149,HARD,25.0,0.48,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.635612,True,FP
1072,2022-03-27:20:25:6417.048000,2022-03-27,20,25,149,HARD,26.0,0.50,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.654635,True,FP
1073,2022-03-27:20:26:6512.557000,2022-03-27,20,26,149,HARD,27.0,0.52,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.656881,True,FP
1074,2022-03-27:20:27:6607.353000,2022-03-27,20,27,149,HARD,28.0,0.54,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.634676,True,FP
1075,2022-03-27:20:28:6702.190000,2022-03-27,20,28,149,HARD,29.0,0.56,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.635262,True,FP


In [8]:
# Pattern 1 follow-up: previous stops of the HARD false alerts (prior_pit_entries comes from dev, a permitted input).
if predictions is not None and 'cases' in globals():
    hard_fp = dev.set_index('opportunity_id').loc[cases.opportunity_id, 'prior_pit_entries']
    print('HARD false alerts by prior_pit_entries:', hard_fp.value_counts().to_dict())

# Pattern 2: copy of the query block for circuit 151 (2022-05-08), the only dev circuit absent from train.
print('Dev circuits absent from train:', sorted(set(dev.circuit_id) - set(train.circuit_id)))
if predictions is not None:
    summary_151, cases_151 = context_query(predictions, 'circuit_id', '151', 'FP')
    display(summary_151)
    print('Alert rate in this race:', summary_151.alerts.iloc[0], '/', summary_151.n.iloc[0],
          '| all dev:', int(predictions.alert.sum()), '/', len(predictions))
    display(cases_151.sort_values('score', ascending=False).head(8))

HARD false alerts by prior_pit_entries: {0: 230}
Dev circuits absent from train: ['151']


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,circuit_id,151,831,46,136,28,621,182,757,74,0.747253,0.179657,0.378378


Alert rate in this race: 182 / 831 | all dev: 938 / 9451


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
3623,2022-05-08:3:23:5956.816000,2022-05-08,3,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.729027,True,FP
3626,2022-05-08:3:26:6242.013000,2022-05-08,3,26,151,MEDIUM,25.0,0.456140,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.715471,True,FP
3305,2022-05-08:11:20:5636.771000,2022-05-08,11,20,151,MEDIUM,19.0,0.350877,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.705372,True,FP
3625,2022-05-08:3:25:6147.017000,2022-05-08,3,25,151,MEDIUM,24.0,0.438596,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.702046,True,FP
4012,2022-05-08:77:21:5739.305000,2022-05-08,77,21,151,MEDIUM,20.0,0.368421,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701659,True,FP
3880,2022-05-08:55:23:5911.352000,2022-05-08,55,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.701486,True,FP
3624,2022-05-08:3:24:6051.816000,2022-05-08,3,24,151,MEDIUM,23.0,0.421053,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.699729,True,FP
3308,2022-05-08:11:23:5919.878000,2022-05-08,11,23,151,MEDIUM,22.0,0.403509,NaN,0.0,forest_sqrt__min_samples_leaf_100,0.697929,True,FP


In [9]:
EVENT_TO_INSPECT = '2022-03-20:18:pit:8372.487000'  # pattern 3 example (early-stint entry)

if predictions is None:
    print('Run your additional candidate first.')
elif EVENT_TO_INSPECT is None:
    print('Set an event_id above when checking a completely missed event.')
else:
    event_windows = positive_windows_for_event(predictions, EVENT_TO_INSPECT)
    display(event_windows)
    if event_windows.empty:
        print('No positive windows found: check the ID; do not call this a missed event.')
    else:
        print('Positive windows:', len(event_windows), 'Alerted:', int(event_windows.alert.sum()))
        print('Completely missed event:', not bool(event_windows.alert.any()))


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
241,2022-03-20:18:42:8042.569000,2022-03-20,18,42,63,MEDIUM,3.0,0.736842,2022-03-20:18:pit:8372.487000,1.0,forest_sqrt__min_samples_leaf_100,0.319648,False,FN
242,2022-03-20:18:43:8142.031000,2022-03-20,18,43,63,MEDIUM,4.0,0.754386,2022-03-20:18:pit:8372.487000,1.0,forest_sqrt__min_samples_leaf_100,0.304343,False,FN


Positive windows: 2 Alerted: 0
Completely missed event: True


In [10]:
# Pattern 3: whole-event check for entries whose positive windows all have tyre_life <= 10 laps.
# Descriptive only: frozen predictions, no refit or threshold change.
if predictions is not None:
    positives = predictions.loc[predictions.y.eq(1)]
    per_event = positives.groupby('event_id').agg(max_tyre_life=('tyre_life', 'max'), caught=('alert', 'any'))
    early = per_event.max_tyre_life.le(10)
    print('Early-stint entries completely missed:', int((~per_event.caught[early]).sum()), '/', int(early.sum()))
    print('Other entries completely missed:', int((~per_event.caught[~early]).sum()), '/', int((~early).sum()))
    print('Median score of their positive windows: early',
          round(positives.loc[positives.event_id.isin(per_event.index[early]), 'score'].median(), 3),
          '| other', round(positives.loc[positives.event_id.isin(per_event.index[~early]), 'score'].median(), 3),
          '| threshold', round(extra_policy['threshold'], 3))

Early-stint entries completely missed: 94 / 102
Other entries completely missed: 117 / 268
Median score of their positive windows: early 0.328 | other 0.596 | threshold 0.602


### What counts as a separate pattern

The same error type in different contexts can count as separate patterns when each entry states its context rule, gives its own count/denominator and traceable case, and explains a consequence and next check. Three cases of the same error in the same context count as one pattern. Changing only a context name without evidence and interpretation is incomplete. You do not need three different causes, independent groups or statistical significance. Declare overlap; do not add overlapping counts as distinct errors.

Fictional examples for interpreting the requirement (these are not Rival results):

- Acceptable entry A: “For context `compound == TOY_A`, 2 of 6 alerts are false (FP/alerts), including `toy-A-4`. These alerts require checking a rival that does not enter within the horizon; next check the neighbouring windows to see whether alerts repeat around the same situation. The six-alert sample is small.”
- Acceptable entry B: “For context `compound == TOY_B`, 3 of 10 alerts are false, including `toy-B-2`. The review burden also occurs here; next inspect the recorded timing around those cases before proposing a cause.” With their separate supporting rows, A and B count as two patterns even though both are FP. Your submission must provide the actual rows behind its own claims.
- Incomplete: “TOY_A has false alerts; TOY_B has false alerts; TOY_C has false alerts.” Context names alone supply neither counts/denominators nor cases, consequences or checks. Three IDs from TOY_A under the same rule also describe one pattern.


### Your three entries

All three use the frozen candidate `forest_sqrt__min_samples_leaf_100` and its dev threshold 0.602. Overall reference on dev: 562 of 938 alerts are false (0.60), and 211 of 370 eligible entries are completely missed. The query settings and outputs are in the cells above: `context-query`, `context-query-pattern2`, `event-query` and `early-stint-events`.

1. **Pattern / rule: false alerts on HARD tyres (window level, `compound == 'HARD'`).**
   - *Evidence count / denominator:* 230 of 274 alerts in this context are false (FP/alerts = 0.84), against 0.60 for all dev alerts. FPR is 230/3,771 negative HARD windows (0.061).
   - *opportunity_id:* `2022-03-27:20:21:6034.524000` (circuit 149, HARD, tyre_life 22, y = 0, score 0.653). It is the first of a run of consecutive false alerts for the same driver.
   - *Observed:* all 230 of these false alerts have `prior_pit_entries = 0`, i.e. drivers still on their first set.
   - *Hypothesis (not measured):* the model learned "old tyre + no stop yet at mid-race = stop soon". Drivers who *start* on hard tyres run a long first stint and break that rule.
   - *Consequence / next check:* the strategist would repeatedly be told that a hard-tyre starter is about to stop when they are not. The alerts repeat lap after lap for the same driver, which wastes attention and could provoke an unnecessary reaction. Next check: compare stint lengths of HARD starters in train (2018–2021) vs dev 2022 (new 18-inch tyres) before attributing a cause.

2. **Pattern / rule: alert concentration at the one circuit not seen in training (window level, `circuit_id == '151'`, race 2022-05-08).**
   - *Evidence count / denominator:* this race receives 182 alerts out of 831 windows (21.9%), against 938/9,451 (9.9%) for all dev. 136 of those 182 alerts are false (FP/alerts = 0.75). FPR is 136/757 negative windows (0.18), against 0.061 for HARD above.
   - *opportunity_id:* `2022-05-08:3:23:5956.816000` (MEDIUM, tyre_life 22, score 0.729, the highest-scoring false alert there).
   - *Observed:* circuit 151 is the only dev circuit with no training rows (`Dev circuits absent from train: ['151']`), so its one-hot circuit columns are all zero.
   - *Hypothesis:* without a circuit-specific signal, the score falls back on tyre age and race progress, which flagged this race far too often.
   - *Overlap:* 72 of the 136 false alerts here are also HARD, so they are counted in pattern 1 too. We do not add the counts.
   - *Consequence / next check:* at a new or heavily changed venue, about one alert in five windows would fire and most would be wrong. That uses up alert attention that should cover the whole season. Next check: compare scores at circuit 151 with windows of the same compound and tyre-age band at seen circuits; treat new venues with caution until data exist.

3. **Pattern / rule: completely missed early-stint pit entries (event level: entries whose positive windows all have `tyre_life <= 10`).**
   - *Evidence count / denominator:* 94 of these 102 entries are completely missed (no positive window alerted). For all other entries, 117 of 268 are missed. All positive windows of each entry were checked, with no context filter.
   - *opportunity_id:* event `2022-03-20:18:pit:8372.487000` (Bahrain, MEDIUM). Both of its positive windows, `2022-03-20:18:42:8042.569000` (tyre_life 3, score 0.320) and `2022-03-20:18:43:8142.031000` (tyre_life 4, score 0.304), are quiet, so `Completely missed event: True`.
   - *Observed:* positive windows of early-stint entries have a median score of 0.328, against 0.596 for other entries, far below the 0.602 threshold. The model essentially never warns about a stop on fresh tyres.
   - *Hypothesis (not measured):* many such stops are reactive (safety car, damage, weather) rather than planned tyre-wear stops. Several entries in this race happen on laps 43–44, which is suggestive but not proof.
   - *Consequence / next check:* the alert is silent exactly for opportunistic stops, which are often the most strategically important. The team must not read "no alert" as "no stop coming". Next check, descriptive only: cross these missed entries with the recorded neutralisation flag (`future_neutralisation`), a future-known field that is *not* an input. This would test the safety-car hypothesis.
   - *Overlap:* patterns 1–2 count false alerts (y = 0 windows) and this pattern counts positive windows, so no window is in both. Four of the 102 early-stint entries occurred at circuit 151.

In [11]:
manifest = export_run(comparison, policies, change, ROOT, predictions=predictions)
print('Additional experiment executed:', manifest['experiment_complete'])
print('This flag does NOT check your reasoning, three patterns, memo or AI record.')
print('Systems:', comparison.model.nunique(), '| train/dev rows:', len(comparison))
print('Generated comparison_table.md and all five outputs/ files. No manual transcription needed.')


Additional experiment executed: True
This flag does NOT check your reasoning, three patterns, memo or AI record.
Systems: 6 | train/dev rows: 12
Generated comparison_table.md and all five outputs/ files. No manual transcription needed.


## 6. Memo and submission

Complete memo.md and PROMPTS.md. The comparison table is generated automatically.
Check CHECKLIST.md, restart + run all, save visible outputs and push the final commit.
Completed comparison: six systems / twelve train-dev rows, plus your additional candidate's diagnostics.
Canvas text entry: both names, accessible repository URL, final commit and notebook path.
The execution flag does not certify your explanation or completed documents.


## Use your feedback next time

Ungraded preparation: revisit one relevant Lab 2 comment, or identify a weakness yourself if no
comment applies. What did you check differently here? Add one or two sentences if useful.
After grading, carry the specific next action from your Lab 3 feedback into the capstone experiment plan.
No extra file, points or oral interview is added by this reflection.
